# Traffic Congestion Classification: Comparing ML Algorithms

**Title:** Comparing Classical Machine Learning and Deep Learning Models for Traffic Congestion Classification on Smart City Sensor Data.

**Research question:** How do classical ML algorithms (SVM, Random Forest, XGBoost, ...) compare with deep learning models (neural networks) for classifying traffic congestion level (Low / Medium / High) from multi-sensor smart-city data, in terms of accuracy, training time, and model size?

---
### How to use this notebook
This notebook is a **template**. Every code cell below is a set of instructions and `# TODO:` comments. **You write the code.** Work top to bottom, one TODO at a time. Ask your instructor if you get stuck.

**Heads up:** the sensor CSVs are small (tens of rows). That is fine for learning the whole ML pipeline, but your numbers will be noisy and the neural networks will tend to overfit. Noticing *why* small data hurts deep learning is part of the project, so keep notes for your report.

# Step 1 - Setup + Download
*(Weeks 1-2)*

In this step you set up your tools, download the traffic dataset, and explore its columns so you understand what you are working with.

## Week 1 - Environment & first look

**Goals:** mount Google Drive, install libraries, download the Metro Interstate Traffic Volume dataset, and load it into a DataFrame.

In [ ]:
# Mount Google Drive so your files are saved between sessions.
# TODO: import the `drive` module from google.colab and call drive.mount('/content/drive')
# TODO: choose a project folder inside your Drive and save its path in a variable, e.g. PROJECT_DIR


In [ ]:
# Install / import the libraries you will need throughout the project.
# TODO: if needed, pip install xgboost (most others are pre-installed on Colab)
# TODO: import numpy, pandas, matplotlib.pyplot
# TODO: import tensorflow and print its version


In [ ]:
# Check what hardware you have (optional - the neural nets here are tiny, so CPU is fine).
# TODO: use tensorflow to list physical GPU devices and print them


In [ ]:
# Download the Metro Interstate Traffic Volume dataset (hourly I-94 traffic + weather, ~48k rows).
# TODO: download https://archive.ics.uci.edu/static/public/492/metro+interstate+traffic+volume.zip
#       and unzip it into PROJECT_DIR (it contains Metro_Interstate_Traffic_Volume.csv.gz)
# TODO: save the path to that .csv.gz file in a variable, e.g. DATA_PATH

In [ ]:
# "Hello data": load the traffic table and look at it.
# TODO: read DATA_PATH into a pandas DataFrame (pd.read_csv reads .gz directly)
# TODO: show the first few rows (.head()) and print its shape (rows, columns)

## Week 2 - Understand the data

**Goals:** understand the columns (traffic volume, weather, time), the weather categories, and how traffic varies by hour of day.

In [ ]:
# Load the dataset and parse the timestamp.
# Columns: holiday, temp (Kelvin), rain_1h, snow_1h, clouds_all, weather_main,
#          weather_description, date_time, traffic_volume.
# TODO: read DATA_PATH into df
# TODO: convert 'date_time' to datetime (pd.to_datetime)
# TODO: print its shape, column names, and df.info()

In [ ]:
# Explore the table.
# TODO: count rows per weather category (value_counts on 'weather_main')
# TODO: count holiday rows (value_counts on 'holiday', dropna=False)
# TODO: describe() the traffic_volume column (min/max/mean)

In [ ]:
# Visualize the readings.
# TODO: histogram of traffic_volume
# TODO: average traffic_volume by hour of day (group by date_time.dt.hour, then plot)

# Step 2 - Data Preprocessing
*(Weeks 3-4)*

Here you clean the data, **add time features (hour, day, month, holiday)**, build a congestion label, split it into train/val/test, and prepare inputs for both classical ML and the neural networks.

## Week 3 - Clean, label & split

**Goals:** drop bad/error readings, parse timestamps, add time features (hour, day, month, holiday), turn traffic volume into a Low/Medium/High label, and split into train/validation/test.

In [ ]:
# Clean the table and parse the timestamp.
# Some rows are sensor errors: temp == 0 (0 Kelvin) and one rain_1h of 9831 mm.
# TODO: convert 'date_time' to datetime
# TODO: drop the error rows: keep temp > 0 and rain_1h < 1000
# TODO: drop duplicate timestamps (drop_duplicates on 'date_time'), then sort by date_time
# TODO: print the cleaned shape

In [ ]:
# Build time features. Traffic depends heavily on the hour and day, so turn the timestamp
# into useful columns. Also convert temperature from Kelvin to Celsius.
# TODO: df['hour']       = date_time.dt.hour
# TODO: df['dayofweek']  = date_time.dt.dayofweek
# TODO: df['month']      = date_time.dt.month
# TODO: df['is_holiday'] = df['holiday'].notna().astype(int)
# TODO: df['temp_c']     = df['temp'] - 273.15
# TODO: print the head of these new columns

In [ ]:
# Build the congestion label from traffic_volume.
# Split the volumes into three equal-sized groups: Low, Medium, High.
# TODO: use pd.qcut(df['traffic_volume'], 3, labels=['Low','Medium','High']) -> df['congestion']
# TODO: print the count of each congestion class

In [ ]:
# Split into train / validation / test (e.g. 60% / 20% / 20%), stratified by congestion.
# "Stratified" keeps the class balance the same in each split.
# TODO: train_test_split TWICE (peel off test, then split the rest into train/val),
#       passing stratify=the congestion label, random_state=42
# TODO: add a column 'split' to df marking each row as train/val/test

## Week 4 - Feature table + scaled arrays

You will prepare data **two ways**:
1. **Classical features** - a table of numbers per reading (weather + time features) for sklearn/XGBoost.
2. **Scaled arrays** - the same features, standardized (mean 0, std 1), which neural networks train on better.

In [ ]:
# Choose the feature columns and one-hot encode the weather category.
# Features: temp_c, rain_1h, snow_1h, clouds_all, hour, dayofweek, month, is_holiday, weather_main.
# (We do NOT use traffic_volume as a feature - it is what the label is built from!)
# TODO: one-hot encode 'weather_main' (pd.get_dummies)
# TODO: build a list FEATURES of the column names you will use

In [ ]:
# Build X (features) and y (label) for each split, then save them.
# TODO: encode the congestion label to integers with LabelEncoder
# TODO: for each split, take X = df[FEATURES][rows in split], y = encoded label
# TODO: np.savez to PROJECT_DIR/features.npz so you don't recompute every time
# TODO: print the shapes of X_train, X_val, X_test


In [ ]:
# Scale the features for the neural networks.
# Neural nets train better when every feature has mean 0 and std 1.
# TODO: fit a StandardScaler on X_train ONLY, then transform train/val/test
# TODO: save the scaled arrays (e.g. features_scaled.npz) for Week 7


In [ ]:
# Quick EDA on the prepared data.
# TODO: confirm the class balance in each split (how many Low/Medium/High per split)
# TODO (optional): correlation heatmap of the numeric features (df[FEATURES].corr())


# Step 3 - Training & Evaluation
*(Weeks 5-8)*

Now you train many models and compare them. Keep ONE shared results table and add a row for every model so the final comparison is easy.

In [ ]:
# Create one shared results table you will append to all through Step 3.
# TODO: make an empty list called `results`
# TODO: write a small helper that, given (name, y_true, y_pred, train_time),
#       computes accuracy and macro-F1 (sklearn.metrics) and appends a dict to results


## Week 5 - Classical models, round 1

Train four simple classifiers on the feature table from Week 4: Logistic Regression, K-Nearest Neighbors, Decision Tree, and Naive Bayes.

In [ ]:
# Load the features you saved in Week 4.
# TODO: np.load PROJECT_DIR/features.npz and pull out X_train, y_train, X_val, y_val, ...


In [ ]:
# Logistic Regression.
# TODO: import LogisticRegression, fit on (X_train, y_train), time the fit
# TODO: predict on X_val and record the result in your results table


In [ ]:
# K-Nearest Neighbors.
# TODO: import KNeighborsClassifier, fit, predict on val, record result
# TODO (variation): try a few values of n_neighbors (e.g. 3, 5) and see what changes


In [ ]:
# Decision Tree.
# TODO: import DecisionTreeClassifier, fit, predict, record result
# TODO (variation): try different max_depth values


In [ ]:
# Naive Bayes.
# TODO: import GaussianNB, fit, predict, record result


In [ ]:
# Show a confusion matrix for your best model so far.
# TODO: use sklearn.metrics.ConfusionMatrixDisplay on val predictions


## Week 6 - Classical models, round 2 + tuning

Train stronger models (SVM, Random Forest, XGBoost) and try **hyperparameter variations** to improve them.

In [ ]:
# Support Vector Machine (SVM).
# TODO: import SVC, train with a linear kernel and again with an 'rbf' kernel
# TODO (variation): try a few values of C; record each result


In [ ]:
# Random Forest.
# TODO: import RandomForestClassifier, fit, predict, record result
# TODO (variation): try different n_estimators and max_depth


In [ ]:
# XGBoost.
# TODO: import XGBClassifier, fit, predict, record result
# TODO (variation): try different n_estimators / max_depth / learning_rate


In [ ]:
# Compare your classical models so far.
# TODO: turn `results` into a DataFrame and sort by accuracy
# TODO: print it and note which classical model is best


## Week 7 - Neural network from scratch

Build your own small neural network (a Multi-Layer Perceptron, or MLP) and train it on the scaled features. Watch what happens with so little data.

In [ ]:
# Load the SCALED features and build a small MLP.
# An MLP stacks Dense layers; the last layer has 3 units + softmax (one score per class).
# TODO: np.load PROJECT_DIR/features_scaled.npz for the scaled X arrays (reuse y from before)
# TODO: build a tf.keras Sequential: Input -> Dense(16, relu) -> Dense(8, relu) -> Dense(3, softmax)
# TODO: compile (optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# TODO: print model.summary()


In [ ]:
# Train the MLP.
# TODO: model.fit(Xs_train, y_train, validation_data=(Xs_val, y_val), epochs=..., verbose=0) and time it
# TODO: keep the History object to plot curves next


In [ ]:
# Plot training curves.
# TODO: plot training vs validation accuracy across epochs
# TODO: plot training vs validation loss across epochs
# Look for overfitting: training accuracy climbing while validation accuracy stalls or drops.


In [ ]:
# Save the model and record its result.
# TODO: model.save to PROJECT_DIR
# TODO: get validation predictions (argmax of model.predict) and append to your results table


## Week 8 - Deeper/wider network + final comparison

Try a bigger neural network with dropout, then compare EVERYTHING. With a small dataset, watch whether the bigger network actually helps or just overfits.

In [ ]:
# Deeper / wider MLP with dropout (dropout randomly turns off neurons to fight overfitting).
# TODO: build a bigger network, e.g. Dense(64) -> Dropout -> Dense(32) -> Dropout -> Dense(3, softmax)
# TODO: train it, time it, record its validation result


In [ ]:
# One more variation of your choice.
# TODO: try a different size, more/fewer epochs, or a different optimizer; record the result


In [ ]:
# FINAL COMPARISON of every model you trained.
# TODO: build the results DataFrame and sort by accuracy / F1
# TODO: bar chart of accuracy per model
# TODO: scatter plot of accuracy vs training time


In [ ]:
# Error analysis on your single best model.
# TODO: pick the best model, predict on the validation set
# TODO: confusion matrix; note which congestion levels get confused with each other


In [ ]:
# Final score: evaluate the best model on the TEST set (do this only once, at the end).
# The test set was never used for training or tuning, so it gives an honest final number.
# TODO: predict on X_test and report accuracy + macro-F1


# Step 4 - Write Report
*(Weeks 9-10)*

Turn your results into a clear story, and build a small demo.

## Week 9 & 10 - Draft & demo

**Goals:** draft the report and build a "predict congestion from a new reading" demo.

### Report draft (write in these markdown cells)

- **Introduction:** Why does predicting traffic congestion in a smart city matter?
- **Dataset:** What is in the Metro Interstate traffic data (traffic volume, weather, time)? How did you clean it and build the label?
- **Methods:** What features and what models did you try?
- **Results:** Your comparison table and charts. Which model won?
- **Discussion:** Why might a simple model beat a neural network here? Which features mattered most (hour of day?)? What are the limits of using a single road sensor? What would you do with more sensors or more cities?

*TODO: write your draft here as you go.*

In [ ]:
# Prediction demo: type in a sensor reading and let your best model guess the congestion level.
# TODO: build one row of features in the SAME column order as FEATURES (e.g. a weekday 8am,
#       clear, mild day) and scale it if your best model is the neural network
# TODO: run best.predict and print the predicted congestion class